# AIPerf Use Case 9 — Custom Prompt Benchmarking (Your Own Prompts, Deterministic Replay)

Benchmark with **your own predetermined prompts** from a JSONL file instead of synthetic token soup or a public trace: every record is sent **exactly as written, exactly once, in file order**. No sampling, no variance between runs — which is what makes results comparable across models, configs, and reruns.

Based on the NVIDIA tutorial [Custom Prompt Benchmarking](https://docs.nvidia.com/aiperf/tutorials/datasets-inputs/custom-prompt-benchmarking); background notes in `docs/reference/aiperf-office-hours.md`.

> **This is the core mechanism of this repo.** Both suites (Model Selection, Sizing) feed hand-written `text_input` prompt files through exactly this pattern — see `model-selection/prompts/content_generation.jsonl` and `docs/scenarios/README.md`. This notebook teaches the pattern itself, standalone.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — building a custom prompt JSONL](#3-input--building-a-custom-prompt-jsonl)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)

## 1. Overview

Three ways to feed AIPerf, and where this one fits:

| Approach | Prompts | Repeatability | Realism |
|---|---|---|---|
| Synthetic (`--synthetic-input-tokens-mean`) | random tokens | perfect | none — no real language, no prefix reuse |
| Public trace (Mooncake, UC3) | expanded block hashes | perfect | production arrival + reuse patterns, but not *your* workload |
| **Custom prompts (this notebook)** | **your exact texts** | **perfect** | **your actual workload** |

Custom prompt benchmarking gives you: **exact control** (each prompt sent verbatim), **deterministic testing** (same requests, same order, every run — sampling variability eliminated), and **production replay** (dump real queries from your app into a JSONL and replay them).

The dataset type is `--custom-dataset-type mooncake_trace` — a naming quirk worth internalizing: here it's just the *file format / replay semantics* (key `text_input`, exact-once, file-order), not the Mooncake dataset. The alternative type `random_pool` (key `text`) samples with replacement instead — see the decision table in Section 3.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). Pin the version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- No external dataset download — the prompt file is built right here in Section 3.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH — install it before running the Test run section.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — building a custom prompt JSONL

One JSON object per line. Two keys matter for the `mooncake_trace` format:

- **`text_input`** (required) — the exact prompt text to send.
- **`output_length`** (optional, per record) — caps generation length for *that* request. Without it, generation runs until the model stops naturally (EOS) or a global `--output-tokens-mean` applies.

```json
{"text_input": "What is the capital of France?", "output_length": 20}
```

Design tips for a good prompt set (mirroring this repo's scenario datasets):

- Make each record a **self-contained test case** representative of one real user task — not filler.
- **Vary input lengths deliberately** if you want to see length-vs-latency behavior in the results.
- Keep the set **small and curated**; exact-once replay means every record runs in every benchmark, so each one should earn its place.

### `mooncake_trace` vs. `random_pool` (per `docs/scenarios/README.md`)

| | `mooncake_trace` (this notebook) | `random_pool` |
|---|---|---|
| JSONL key | `text_input` | `text` / `texts` |
| Selection | deterministic file-order replay, exactly once each | `--dataset-sampling-strategy` (`sequential` / `random` / `shuffle`), with replacement |
| Run length | = number of records | `--request-count` or `--benchmark-duration` |
| Best for | comparison runs (models, configs, concurrency rungs) — every run sees identical requests | sustained/soak — continuous varied traffic from a small pool over a long run |

This repo's Model Selection & Sizing suites use `mooncake_trace` (comparability), while the Sustained/Soak scenario uses `random_pool` — the two are not interchangeable: switching the key breaks AIPerf's schema validation, and switching the semantics breaks the test's logic.

In [ ]:
# A small curated prompt set: content-generation-style tasks with deliberately
# varied input lengths, two of them with a per-record output_length cap.
prompts = [
    {"text_input": "What is the capital of France?", "output_length": 20},
    {"text_input": "List three practical benefits of database connection pooling.", "output_length": 100},
    {"text_input": "Write a two-sentence product announcement for a mobile app "
                   "that tracks personal carbon footprint."},
    {"text_input": "Draft a short, friendly email to a customer explaining that their "
                   "reported bug has been fixed in today's release and how to update."},
    {"text_input": "Summarize the trade-offs between monolithic and microservices "
                   "architectures for a five-person startup engineering team, covering "
                   "deployment complexity, iteration speed, and operational cost."},
    {"text_input": "You are a technical writer. Given the following changelog entries, write "
                   "release notes for end users. Entries: 1) Fixed race condition in job "
                   "scheduler causing duplicate runs under high load. 2) Added exponential "
                   "backoff to webhook retries (max 5 attempts). 3) Dashboard latency chart "
                   "now supports p50/p90/p99 toggles. 4) Deprecated the v1 export API, "
                   "removal planned next quarter. 5) Reduced cold-start time of serverless "
                   "workers by ~40% via snapshot restore."},
]

prompt_path = REPO_ROOT / "notebooks" / "data" / "custom_prompts.jsonl"
prompt_path.parent.mkdir(parents=True, exist_ok=True)
with open(prompt_path, "w", encoding="utf-8") as f:
    for p in prompts:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Wrote {len(prompts)} prompts -> {prompt_path}")
for i, p in enumerate(prompts):
    cap = f" (output_length={p['output_length']})" if "output_length" in p else ""
    print(f"  [{i}] {len(p['text_input'].split()):>3} words{cap}: {p['text_input'][:60]}...")


## 4. Test run

The dataset defines the run: **no `--request-count`** — each record is sent once, so the run is `len(prompts)` requests (warm-up requests are extra and excluded from metrics).

Concurrency choice matters for interpretation:

- **`CONCURRENCY = 1`** — requests run strictly sequentially in file order, so per-request results map 1:1 to prompt file lines (best for per-prompt debugging; also this repo's model-selection baseline).
- **Higher concurrency** — requests dispatch in file order but overlap, so per-prompt latencies include queueing effects (realistic, less attributable). The tutorial uses `2`.

In [ ]:
def run_aiperf(cmd):
    """Print and run an aiperf command from the repo root, streaming output into the notebook."""
    print("$ " + " \\\n    ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\nexit code: {result.returncode}")
    return result

# ---- Required ----------------------------------------------------------
MODEL = ""       # model name the endpoint serves, e.g. "Qwen/Qwen3-0.6B"
URL = ""         # e.g. "http://localhost:8000"

# ---- Optional -----------------------------------------------------------
TOKENIZER = ""            # empty = use MODEL
CONCURRENCY = 1           # 1 = sequential, per-prompt attributable (see above)
WARMUP_REQUESTS = 1
# NOTE: --output-tokens-mean is the canonical long name for the gist's --osl
# shorthand. Empty = no global cap: records with "output_length" are capped
# per-record, the rest generate until the model stops naturally.
OUTPUT_TOKENS_MEAN = ""
OUTPUT_DIR = "artifacts/aiperf-uc9-custom-prompts"

assert MODEL and URL, "Set MODEL and URL above."

cmd = [
    "aiperf", "profile",
    "--model", MODEL,
    "--url", URL,
    "--endpoint-type", "chat",
    "--streaming",
    "--input-file", str(prompt_path),
    "--custom-dataset-type", "mooncake_trace",
    "--concurrency", str(CONCURRENCY),
    "--warmup-request-count", str(WARMUP_REQUESTS),
    "--tokenizer", TOKENIZER or MODEL,
    "--artifact-dir", OUTPUT_DIR,
]
if OUTPUT_TOKENS_MEAN:
    cmd += ["--output-tokens-mean", str(OUTPUT_TOKENS_MEAN)]
run_aiperf(cmd)


## 5. Results analysis

With a small exact-once dataset, the per-request export is more informative than the aggregate summary — six curated prompts deserve six rows, not one average. Both views below.

In [ ]:
import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
csv_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
assert csv_path is not None, f"No profile_export_aiperf.csv under {artifact_dir} — did the Test run complete?"

stats = pd.read_csv(csv_path)
metric_col = stats.columns[0]
key = ["time to first token", "request latency", "inter token latency",
       "input sequence length", "output sequence length", "request count", "error"]

# Per-request statistics section, filtered to the key metrics — one row per
# metric, columns are avg/min/max/percentiles/std aggregated across all 6 prompts.
stats[stats[metric_col].str.lower().apply(lambda n: any(p in n for p in key))]


In [ ]:
# Per-request view: one row per prompt. Sorted by request start time — with
# CONCURRENCY = 1 that IS the prompt file order, so row i = prompt i.
jsonl_path = next(artifact_dir.rglob("profile_export.jsonl"), None)
assert jsonl_path is not None, f"No profile_export.jsonl under {artifact_dir}."

records = []
with open(jsonl_path) as f:
    for line in f:
        rec = json.loads(line)
        row = {"request_start_ns": rec.get("metadata", {}).get("request_start_ns")}
        for name, m in rec.get("metrics", {}).items():
            row[name] = m["value"] if isinstance(m, dict) and "value" in m else m
        records.append(row)

per_request = pd.DataFrame(records).sort_values("request_start_ns").reset_index(drop=True)
if CONCURRENCY == 1 and len(per_request) == len(prompts):
    per_request.insert(0, "prompt", [p["text_input"][:50] + "..." for p in prompts])

cols = [c for c in ["prompt", "input_sequence_length", "output_sequence_length",
                    "time_to_first_token", "inter_token_latency", "request_latency"]
        if c in per_request.columns]

# One row per prompt (in file order at CONCURRENCY = 1) — ISL/OSL and latency
# metrics for that exact prompt, not aggregated across the run.
per_request[cols]


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if "request_latency" in per_request.columns:
    per_request["request_latency"].plot.barh(ax=axes[0])
    axes[0].set_title("Request latency per prompt (file order)")
    axes[0].set_xlabel("ms")
    axes[0].set_ylabel("prompt #")

if {"input_sequence_length", "time_to_first_token"} <= set(per_request.columns):
    axes[1].scatter(per_request["input_sequence_length"], per_request["time_to_first_token"])
    axes[1].set_title("TTFT vs. input length")
    axes[1].set_xlabel("ISL (tokens)")
    axes[1].set_ylabel("TTFT (ms)")

plt.tight_layout()


### Notes

- **Reruns are directly comparable**: same file, same order, same requests — any metric delta between two runs is the endpoint's doing, not the workload's. That determinism is why this repo's comparison suites use this pattern (and why the soak scenario, which wants variety over hours, does not — see the Section 3 table).
- **Watch `output_sequence_length` when there's no cap**: prompts without `output_length` stop at EOS, so OSL varies across models — two models answering the same prompt with different verbosity will show different request latencies even at identical tokens/sec. Cap outputs (per-record or `--output-tokens-mean`) when you want latency numbers isolated from verbosity differences.
- **Production replay**: dump real user queries into this JSONL shape (`text_input` per line) and you're replaying your actual workload; add timestamps and `--fixed-schedule` if you also want the real arrival pattern (see the UC3 notebook).
- The repo's scenario prompt files (`model-selection/prompts/content_generation.jsonl`, `sizing/`) are production-grade examples of this exact format — curated one-shot test cases, no `output_length` (OSL is controlled globally by the scenario scripts instead).